# BCC API Search Experimentation

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [25]:
import os
import json
from requests_oauth2client import OAuth2Client, OAuth2ClientCredentialsAuth
import swagger_client as bcc_api_client

# Initialize OAuth2 client for BCC and client-credentials auth (same as backend/app.py)
bcc_oauth_client = OAuth2Client(
    token_endpoint="https://login.bcc.no/oauth/token",
    client_id=os.environ["BCC_OAUTH_CLIENT_ID"],
    client_secret=os.environ["BCC_OAUTH_CLIENT_SECRET"],
)

bcc_auth = OAuth2ClientCredentialsAuth(
    bcc_oauth_client,
    scope="persons.name#read",
    audience="api.bcc.no",
)

# Configure API client
bcc_client_config = bcc_api_client.Configuration()
bcc_client_config.host = "https://core.api.bcc.no"
api_client = bcc_api_client.ApiClient(configuration=bcc_client_config)
persons_api = bcc_api_client.PersonsApi(api_client)

if bcc_auth.token is None or bcc_auth.token.is_expired():
    bcc_auth.renew_token()
persons_api.api_client.configuration.access_token = str(bcc_auth.token)

In [31]:
def print_names(result):
    print(f"Found {len(result.data)} persons:")
    for person in result.data:
        print("  ", person.display_name)

In [32]:
# Example 1: _contains is not case-sensitive - search returns no results
print_names(persons_api.find_persons(filter=json.dumps({"displayName": {"_contains": "jacob"}})))

Found 0 persons:


In [ ]:
# Example 2: No results for "jacob", even though several persons named "Jacob" exists
print_names(persons_api.find_persons(search="jacob"))
print_names(persons_api.find_persons(fields="*", search="jacob e")) # found him

Found 0 persons:
Found 1 persons:
   Jacob Eisenberg


In [ ]:
# Example 3: "Ida Sophie Merkle-Børsting" is very hard to find
print_names(persons_api.find_persons(search="ida"))
print_names(persons_api.find_persons(search="ida m"))
print_names(persons_api.find_persons(search="ida b"))
print_names(persons_api.find_persons(search="ida m b"))
print_names(persons_api.find_persons(search="ida børsting"))  # finally...
print_names(persons_api.find_persons(search="ida merkle"))  # also works
print_names(persons_api.find_persons(search="ida merkle børsting"))  # also works

Found 1 persons:
   Ida Puls
Found 1 persons:
   Ida Puls
Found 1 persons:
   Ida Puls
Found 1 persons:
   Ida Puls
Found 14 persons:
   Mary Børsting
   Ida Sophie Merkle-Børsting
   Kirsten Børsting
   Lissen Børsting
   Bella Noella Børsting
   Jens Otto Børsting
   Theis Børsting
   Aurora Gry Børsting
   Colin Riis Børsting
   Svend Aage Børsting
   Terkel Børsting
   Helle Møller
   Grethe Børsting
   Valde Hubert Børsting
Found 1 persons:
   Ida Sophie Merkle-Børsting
Found 9 persons:
   Ida Sophie Merkle-Børsting
   Mary Børsting
   Helle Møller
   Kirsten Børsting
   Lissen Børsting
   Terkel Børsting
   Bella Noella Børsting
   Theis Børsting
   Jens Otto Børsting


In [ ]:
# Example 4: We want to find a person who changed name from "Sven Nielsen" to "Sven Kirkegaard":
print_names(persons_api.find_persons(search="sven"))
print_names(persons_api.find_persons(search="sven n"))
print_names(persons_api.find_persons(search="sven nielsen")) # lots of other Nielsen's
print_names(persons_api.find_persons(search="sven k")) # no results
print_names(persons_api.find_persons(search="sven kirkegaard")) # found him

Found 0 persons:
Found 0 persons:
Found 23 persons:
   Steen Nielsen
   Steffen Nielsen
   Daniel Nielsen
   Timmy Nielsen
   Preben Nielsen
   David Nielsen
   Laura Nielsen
   Danny Nielsen
   Allan Nielsen
   Natalie Nielsen
   Joachim Nielsen
   Bo Johan Nielsen
   Karen Erz Nielsen
   Kristian Nielsen
   Severin Riis Nielsen
   Alexandra Nielsen
   Oscar Nielsen
   Jens Peter Nielsen
   Ellida Nielsen
   Vibeke Marie Nielsen
   Katrine  Riis Nielsen
   Niels Kristensen
   Ester Nielsen
Found 0 persons:
Found 8 persons:
   Sven Adler Kirkegaard
   Eva Riis Kirkegaard
   Frank Kirkegaard Nielsen
   Kaj Erik Kirkegaard Nielsen
   Mila Kirkegaard Bourcier
   Carina Riis Kirkegaard Nielsen
   Birgitte Bourcier
   Jean-Louis Kirkegaard Bourcier
